<a href="https://colab.research.google.com/github/JJJuniorDev/ML-colab/blob/main/tf_keras_batchNorm_callbacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras.datasets import cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = (x_train - x_train.mean()) / x_train.std()
x_test  = (x_test  - x_train.mean()) / x_train.std()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
from functools import partial
import tensorflow as tf
RegularizedDense=partial(tf.keras.layers.Dense,
                         activation=None,  # rimuoviamo qui
                         kernel_initializer="lecun_normal",
                         kernel_regularizer=tf.keras.regularizers.l2(1e-4))

In [ ]:

model=tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(32,32,3)),

    RegularizedDense(100, use_bias=False),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    RegularizedDense(100, use_bias=False),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    RegularizedDense(100, use_bias=False),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    RegularizedDense(100, use_bias=False),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    RegularizedDense(100, use_bias=False),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    tf.keras.layers.Dense(100, use_bias=False, kernel_initializer="he_normal"),
    #tf.keras.layers.BatchNormalization(),
    tf.keras.layers.AlphaDropout(rate=0.2),
    tf.keras.layers.Activation('selu'),

    tf.keras.layers.Dense(10, activation="softmax")  # output finale
])

optimizer=tf.keras.optimizers.Nadam()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)


In [ ]:
model.fit(x_train, y_train, epochs=30, validation_data=(x_test, y_test), callbacks=[early_stop, lr_schedule])

Epoch 1/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.2249 - loss: 2.4112 - val_accuracy: 0.2278 - val_loss: 2.4250 - learning_rate: 0.0010
Epoch 2/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.3555 - loss: 1.8480 - val_accuracy: 0.2379 - val_loss: 2.7868 - learning_rate: 0.0010
Epoch 3/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.3924 - loss: 1.7853 - val_accuracy: 0.2893 - val_loss: 2.4692 - learning_rate: 0.0010
Epoch 4/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.4150 - loss: 1.7440 - val_accuracy: 0.2377 - val_loss: 3.1266 - learning_rate: 0.0010
Epoch 5/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.4416 - loss: 1.6788 - val_accuracy: 0.2898 - val_loss: 2.5650 - learning_rate: 5.0000e-04
Epoch 6/30
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.4551 - loss: 1.6291 - val_accuracy: 0.2231 - val_loss: 3.9519 - learning_rate: 5.0000e-04


In [ ]:
import numpy as np
def mc_dropout_predict(model, x, n_samples=100):
    preds = [model(x, training=True).numpy() for _ in range(n_samples)]
    preds = np.array(preds)
    return preds.mean(axis=0), preds.std(axis=0)

In [ ]:
mean_pred, uncertainty = mc_dropout_predict(
    model,
    x_test[:1],
    n_samples=100
)